# Setting Up the Environment

In [257]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

original_data = pd.read_csv('./dataset1.csv', parse_dates=True) # reading data from csv file
data = original_data.copy()  # making a copy so we don't modify the original dataset
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 907 entries, 0 to 906
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   start_time                 907 non-null    object 
 1   bat_landing_to_food        907 non-null    float64
 2   habit                      866 non-null    object 
 3   rat_period_start           907 non-null    object 
 4   rat_period_end             907 non-null    object 
 5   seconds_after_rat_arrival  907 non-null    int64  
 6   risk                       907 non-null    int64  
 7   reward                     907 non-null    int64  
 8   month                      907 non-null    int64  
 9   sunset_time                907 non-null    object 
 10  hours_after_sunset         907 non-null    float64
 11  season                     907 non-null    int64  
dtypes: float64(2), int64(5), object(5)
memory usage: 85.2+ KB


### Checking for any null value

In [258]:
data.isna().sum()

start_time                    0
bat_landing_to_food           0
habit                        41
rat_period_start              0
rat_period_end                0
seconds_after_rat_arrival     0
risk                          0
reward                        0
month                         0
sunset_time                   0
hours_after_sunset            0
season                        0
dtype: int64

### Checking for any duplicates

In [259]:
data.duplicated().sum() # checking for duplicate

np.int64(1)

# Calculating and Removing Outlier

### Function to detect outlier

In [260]:
def detect_outlier(col):
   num = pd.to_numeric(col, errors='coerce') # Making sure all the values are numeric.
   q1 = num.quantile(0.25) # calculating first quantile
   q3 = num.quantile(0.75) # calcualting third quantile
   IQR = q3 - q1 # calculating Interquartile range
   lo = q1 - (1.5 * IQR) # calculating lower fence for data, a potential outlier.
   hi = q3 + (1.5 * IQR) # calculating high fence for data, a potential outlier
   outlier = (num < lo) | (num > hi)
   return outlier  # returns boolen value


### Flagging outlier from given column

We are flagging the outlier, if we remove the outlier then it might create an data imbalance

In [261]:
col_name = ['bat_landing_to_food', 'seconds_after_rat_arrival'] #column where outlier we are flagging
def flag_outlier(col_name):
    for col in col_name:
        outlier = detect_outlier(data[col])  # detecting the outliers row in a given col
        data.loc[outlier, col] = 0 # flagging specific row for given column
    return data
cdata = flag_outlier(col_name)
cdata.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 907 entries, 0 to 906
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   start_time                 907 non-null    object 
 1   bat_landing_to_food        907 non-null    float64
 2   habit                      866 non-null    object 
 3   rat_period_start           907 non-null    object 
 4   rat_period_end             907 non-null    object 
 5   seconds_after_rat_arrival  907 non-null    int64  
 6   risk                       907 non-null    int64  
 7   reward                     907 non-null    int64  
 8   month                      907 non-null    int64  
 9   sunset_time                907 non-null    object 
 10  hours_after_sunset         907 non-null    float64
 11  season                     907 non-null    int64  
dtypes: float64(2), int64(5), object(5)
memory usage: 85.2+ KB


# Analysing Habit Column

In [262]:
data['habit'].isna().sum()  # calcualting the number of row that contains NaN value.
data['habit'].unique()  # analyzing the uniqueness of data in habit column
data['habit'].value_counts()


habit
fast                                                245
rat                                                 221
pick                                                139
bat                                                  30
bat_fight                                            26
                                                   ... 
bat_fight_and_rat                                     1
rat_and_rat                                           1
not_sure_rat                                          1
501.0,358.4,636.2,423.4; 476.0,103.0,634.0,206.0      1
rat_and_bat_and_pick                                  1
Name: count, Length: 81, dtype: int64

### Cleaning Junks from Habit Column
After analyzing habit column, we knew it contains junk data and some strings contains characters like 'rat_attack'. At this stage, we don't know how important habit column. So we will be replacing the junk data and NaN value with `Unknown` string.

In [263]:
def clean_habit(row):
    if pd.isna(row):    #checking if given value is null value or not
        return 'Unknown'
    val_check = str(row).strip()  # converting everything to strings
    
    if re.fullmatch(r'^[\d\.,;\s]+$', val_check): # checks if strings is made of digit, dots or character
        return 'Unknown'
    
    val = val_check.lower().replace('_', ' ')  # lower strings and replacing '_' with white space
    return re.sub(r'\s+', ' ', val).strip()

data.loc[:,'habit'] = data['habit'].map(clean_habit)  # assigning cleaned column to copied df


In [264]:
data['habit'].value_counts()

habit
fast                    245
rat                     221
pick                    139
Unknown                  58
bat                      30
                       ... 
fight bat                 1
bat fight and rat         1
rat and rat               1
not sure rat              1
rat and bat and pick      1
Name: count, Length: 64, dtype: int64

### Grouping data with less frequency 
After removing the junk, we can see there are few data with less frequency. We decided to grouping them into an 'other' category instead of removing them. This approach prevents data loss while simplifying the dataset.

In [265]:
def categorize_low_frequency(col, threshold: int):
    counts = col.value_counts() # calculate the frequency of each unique string in the series.
    low_freq = counts[counts < threshold].index.tolist() # identify values with a frequency less than given thresold
    cleaned_data = data['habit'].replace(low_freq, 'Other')
    return cleaned_data

cat_habit =  categorize_low_frequency(data['habit'], 3)
data.loc[:, 'habit'] = cat_habit

# Converting to datetime
We have some columns that contain date and time as data, however, their dtype is object. We are trying to conver them into datetime. 

In [266]:
cat_cols = ['start_time', 'rat_period_start', 'rat_period_end', 'sunset_time'] #colmns that contain data in {date time} format.
for col in cat_cols:  # Converting dtype object in to dtype datetime.
    data[col] = pd.to_datetime(data[col], dayfirst = True)

# Creating new Column called Date
data['Date'] = data['start_time'].dt.date

# Converting T.D. in Second and Hour into same Unit.
Few columns contains the time difference between two events in second, where as other columns contains the time differnce in hour. For our analyis, we are converting them into same unit i.e Minute.

In [275]:
# Converting bat_landing_to_food in min
data['bat_landing_to_food'] = data['bat_landing_to_food'] / 60

#converting hour_after_sunset in min
data['hours_after_sunset'] = data['hours_after_sunset'] * 60

#calculating TD. in minutes after sunset and before bat landing to food platform
data['minutes_after_sunset'] = (data['start_time'] - data['sunset_time']).dt.total_seconds() / 60
data['minutes_after_sunset']

0      112.0
1      186.0
2      186.0
3      187.0
4      189.0
       ...  
902    536.0
903    536.0
904    537.0
905    269.0
906    617.0
Name: minutes_after_sunset, Length: 907, dtype: float64

### Performing Chronological check

In [ ]:
data['chronology_check'] = (data['rat_period_start'] <= data['start_time']) & (data['start_time'] <= data['rat_period_end'])


0      True
1      True
2      True
3      True
4      True
       ... 
902    True
903    True
904    True
905    True
906    True
Name: chronology_check, Length: 907, dtype: bool